# stitch together ocean boundary nudging files of different sources into a one file each for salt and temp

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
path2nudging="/expanse/lustre/scratch/jisrael/temp_project/run_schism/nudging_2026_03_03/"

In [3]:
ext1="cencoos"
ext2="hycom"

In [4]:
#salinity nc files first 
file1=xr.open_dataset(path2nudging+'SAL_nu_'+ext1+'.nc')
file2=xr.open_dataset(path2nudging+'SAL_nu_'+ext2+'.nc')
file2

<xarray.Dataset> Size: 3GB
Dimensions:               (node: 1368, time: 24000, nLevels: 23, one: 1)
Coordinates:
  * time                  (time) float32 96kB 0.0 3.6e+03 ... 8.639e+07 8.64e+07
Dimensions without coordinates: node, nLevels, one
Data variables:
    map_to_global_node    (node) int32 5kB ...
    tracer_concentration  (time, node, nLevels, one) float32 3GB ...

In [5]:
# first check do the files have the same number of dimensions? Are all but time the same?
print(file1.dims)
print(len(file1.dims))
flag=0
if len(file1.dims) == len(file2.dims):
    for d in file1.dims:
        if d != 'time':
            print('Checking dimension '+d+'...')
            if d in file2.dims:
                if len(file1[d])==len(file2[d]):
                    print('This dim is good to concat!')
                else:
                    print('Dimension '+d+' is not the same size in the 2 files.')
                    flag=flag+1
            else:
                print('The two files do not have the same dimension names')
                flag=flag+1
        
else:
    print('Error: Files have different numbers of dimensions.')
    flag=flag+1


FrozenMappingWarningOnValuesAccess({'node': 1368, 'time': 3480, 'nLevels': 23, 'one': 1})
4
Checking dimension node...
This dim is good to concat!
Checking dimension nLevels...
This dim is good to concat!
Checking dimension one...
This dim is good to concat!


In [6]:
if flag==0:
    file2save=xr.concat([file1,file2],dim="time")

file2save

/scratch/jisrael/job_46989216/ipykernel_219638/3010770711.py:2: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  file2save=xr.concat([file1,file2],dim="time")


<xarray.Dataset> Size: 4GB
Dimensions:               (time: 27480, node: 1368, nLevels: 23, one: 1)
Coordinates:
  * time                  (time) float32 110kB 0.0 3.6e+03 ... 8.64e+07
Dimensions without coordinates: node, nLevels, one
Data variables:
    map_to_global_node    (time, node) int32 150MB 36568 36569 ... 50759 51120
    tracer_concentration  (time, node, nLevels, one) float32 3GB 33.51 ... 33.61

In [7]:
#write the file to a desired location
writepath=path2nudging+"SAL_nu_combinded.nc"
file2save.to_netcdf(writepath)

In [9]:
def stitch_nudging(exts,path2nudging):
    #where exts is a vector of strings of the extensions in your files you want to splice together, e.g. ['cencoos','hycom']
    nudgedvars=['TEM','SAL']
    for v in nudgedvars:
        filelist=[]
        for x in exts:
            file=xr.open_dataset(path2nudging+v+'_nu_'+ext1+'.nc')
            filelist.append(file)
        flag=0
        #inside var loop
        if len(set([len(f.dims) for f in filelist]))==1:
            print('Files have same number of dimensions!')
            for d in filelist[0].dims:
                if d != 'time':
                    print('Checking dimension '+d+'...')
                    if d in file2.dims:
                        if len(file1[d])==len(file2[d]):
                            print('This dim is good to concat!')
                        else:
                            print('Dimension '+d+' is not the same size in the 2 files.')
                            flag=flag+1
                    else:
                        print('The two files do not have the same dimension names')
                        flag=flag+1
                
            else:
                print('Error: Files have different numbers of dimensions.')
            flag=flag+1
                
                
        

In [10]:
exts=['cencoos','hycom']
nudgedvars=['TEM','SAL']
for v in nudgedvars:
    filelist=[]
    for x in exts:
        file=xr.open_dataset(path2nudging+v+'_nu_'+ext1+'.nc')
        filelist.append(file)
    flag=0
    #inside var loop
    if len(set([len(f.dims) for f in filelist]))==1:
        print('same num of dims!')

same num of dims!
same num of dims!


In [16]:
if [list(f.dims) for f in filelist]:
    print(f.dims.keys)

TypeError: unhashable type: 'list'

In [17]:
list(filelist[0].dims)

['node', 'time', 'nLevels', 'one']

In [20]:
filelist.dims

AttributeError: 'list' object has no attribute 'dims'

# UPDATE DO NOT need to concatenate the gr3s they do not have time history Now try the gr3s

In [15]:
# I think need to skip the first 2 lines, add that back as a header later
header3=pd.read_csv(path2nudging+'SAL_nudge_'+ext1+'.gr3',header=None,nrows=2)
header3

,0
0,bay_delta_116
1,563617 468236 ! # of elements and nodes


In [24]:
header3.iloc[:,0]

0                              bay_delta_116
1    563617 468236 ! # of elements and nodes
Name: 0, dtype: str

In [17]:
file3=pd.read_csv(path2nudging+'SAL_nudge_'+ext1+'.gr3',skiprows=2,header=None)
file3

,0
0,1 611016.00000000 4295635.74000000 ...
1,2 611016.92500000 4295644.09000000 ...
2,3 611014.50000000 4295618.18000000 ...
3,4 611050.75700000 4295635.31000000 ...
4,5 611050.71200000 4295642.37000000 ...
...,...
1031848,563613 3 96855 97647 97258
1031849,563614 3 72426 72674 72424
1031850,563615 3 72426 72424 72172
1031851,563616 3 278918 279173 279172


In [25]:
file4=pd.read_csv(path2nudging+'SAL_nudge_'+ext2+'.gr3',skiprows=2,header=None)
file4

,0
0,1 611016.00000000 4295635.74000000 ...
1,2 611016.92500000 4295644.09000000 ...
2,3 611014.50000000 4295618.18000000 ...
3,4 611050.75700000 4295635.31000000 ...
4,5 611050.71200000 4295642.37000000 ...
...,...
1031848,563613 3 96855 97647 97258
1031849,563614 3 72426 72674 72424
1031850,563615 3 72426 72424 72172
1031851,563616 3 278918 279173 279172


In [ ]:
for d in file1.dims:
    print(d)

In [ ]:
file1.node

In [ ]:
flag